## Topic: RunnablePassthrough

### Agenda
- 1. Introduction of RunnablePassthrough

- 2. Workflow

- 3. Practical Example

### 1. Introduction of RunnablePassthrough

- Definition:
    - RunnablePassthrough is a special Runnable primitive that simply return the input as output without modifying it.



- input (x) --> RunnablePassthrough ---> x (no change)

In [ ]:
""" 

            RunnableLambda
                ↓
            Transform the input
            x → f(x)


            RunnablePassthrough
                ↓
            Keep the input
            x → x


            RunnableParallel
                ↓
            Run independent branches
            x → {A(x), B(x), C(x)}

"""


### 2. Workflow

In [ ]:





"""         - RAG Architecture 

                         User Question
                               │
                 ┌─────────────┴─────────────┐
                 │                           │
                 ↓                           ↓
             Retriever             RunnablePassthrough
                 │                           │
                 ↓                           ↓
             Context                    Question
                 │                           │
                 └─────────────┬─────────────┘
                               ↓
                            Prompt
                               ↓
                              LLM
                               ↓
                             Parser
                               ↓
                            Answer


"""




### 3. Practical Example

In [ ]:
# Example 3.1: RunnablePassThrough
# Purpose -> How to work RunnablePassThrough

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassThrough

from dotenv import load_dotenv
load_dotenv()

passthrough = RunnablePassThrough()

response = passthrough.invoke({
    "name": "KzRaihan"
})

print(response)

- key note:
    - answer : KzRaihan (as input)

#### Example: 3.2

- Generate joke based on topic and explain the joke {topic} and also give the original text.

- Idea:
    - step1: 
        - prompt1:  Generate a joke base on {topic} 

    - step2: RunnableSequence
        - input: prompt1
        - process: RunnableSequence
        - output: joke on {topic}


    - step3: RunnableParallel 
        - input: Prompt2-> Explain the joke on {topic}
        - process: RunnableParallel
                        -> Prompt2 -> RunnableSequence -> parser
                        -> prompt1 -> RunnablePassThrough ---> parser
        
        
        - output: 
            - Generate a joke about the topic and explain the joke.
            - Generate the original Text



In [ ]:
# Example 1: RunnablePassThrough

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassThrough

from dotenv import load_dotenv
load_dotenv()


# Define prompt1 -> write a joke on topic
prompt1 = PromptTemplate(
    template = "write a joke about {topic}",
    input_variables=["topic"]
)

# prompt2 -> Explain the joke on topic
prompt2 = PromptTemplate(
    template = "Explain the following joke - \n{topic}",
    input_variables=["topic"]
)


# define model
model = ChatOpenAI()

# Define parser
parser = StrOutputParser()

# Create a  chain using  (LCEL)
joke_gen_chain = RunnableSequence(prompt1 | model | parser)

# parallel chain for -> 1. original text, 2. explain the joke
parallel_chain = RunnableParallel({
    'joke': RunnablePassThrough(),
    "explain": RunnableSequence(prompt2 | model | parser)

})

# final chain for connection between joke_gen_chain and parallel_chain
final_chain = RunnableSequence(joke_gen_chain | parallel_chain)


# response
response = chain.invoke(
    {
        "topic": "GenAI"
    }
)

print(f"Response: \n {response}")

In [ ]:
### Practical RAG Code Structure
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough
)


# ---------------------------------------------------------
# Step 1: Create the retrieval + question preparation
# ---------------------------------------------------------

retrieval_chain = RunnableParallel({
    # Retrieve relevant documents
    "context": retriever,

    # Keep the original user question unchanged
    "question": RunnablePassthrough()
})


# ---------------------------------------------------------
# Step 2: Connect the retrieved information to the prompt
# ---------------------------------------------------------

rag_chain = retrieval_chain | prompt | model | parser


# ---------------------------------------------------------
# Step 3: Ask a question
# ---------------------------------------------------------

answer = rag_chain.invoke(
    "What is machine learning?"
)


print(answer)